# 07 · Compare priorities and expose assumptions

**45 minutes · Intermediate.** Build a transparent exercise score, test weight sensitivity, and create a decision log with owners and verification steps.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=07_EOC_Planning.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## A score supports discussion; it does not authorize action
We combine three fictional signals with explicit weights: reported burden (50%), flood exposure (30%) and access delay (20%). Fixed classroom anchors are 300 reports per 100,000, 100% exposed population and 24 hours of delay. Each component is clipped to 0–1, then the weighted sum is multiplied by 100.
These anchors and weights are **not calibrated, validated, or recommended operational thresholds**. The algorithm excludes information quality from the score and displays it separately. A real planning meeting also needs local knowledge, feasibility, equity, existing commitments and responsible authority.

In [ ]:
from fieldguide import priority_scores
s=summarize()
ranked=priority_scores(s)
display(ranked[["district","exercise_score","completeness_pct","review_needed","usable_beds"]].round(1))

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
left=np.zeros(len(ranked))
for component,weight in {"reported_burden":.5,"flood_exposure":.3,"access_delay":.2}.items():
    width=ranked[component].to_numpy()*weight*100
    ax.barh(ranked.district,width,left=left,label=component.replace("_"," "))
    left+=width
ax.invert_yaxis(); ax.set(xlabel="Exercise score (0–100)",title="SYNTHETIC · Contributions to a discussion score")
ax.legend(loc="lower right"); plt.show()

## Test sensitivity before defending a rank
Reweighting access changes the question being answered. Display both rankings. A stable rank under these two choices is not proof of a valid model.

In [ ]:
alternative=priority_scores(s,{"reported_burden":.2,"flood_exposure":.3,"access_delay":.5})
rank_a=ranked.reset_index(drop=True).assign(baseline_rank=lambda x:x.index+1)
rank_b=alternative.reset_index(drop=True).assign(access_rank=lambda x:x.index+1)
comparison=rank_a[["district","baseline_rank"]].merge(rank_b[["district","access_rank"]],on="district")
display(comparison)
fig,ax=plt.subplots(figsize=(9,5))
for r in comparison.itertuples():
    ax.plot([0,1],[r.baseline_rank,r.access_rank],"o-",alpha=.6)
    ax.text(-.03,r.baseline_rank,r.district,ha="right",fontsize=9)
ax.set_xticks([0,1],["Baseline weights","More weight on access"])
ax.set(ylabel="Rank (1 first)",title="SYNTHETIC · Sensitivity of discussion order",xlim=(-.5,1.2))
ax.invert_yaxis(); plt.show()

In [ ]:
log=pd.DataFrame([
    {"district":r.district,"proposed_step":"Verify need and access with district focal point",
     "owner":"Planning lead","due_utc":"2025-11-29 12:00","status":"EXERCISE — pending verification",
     "reason":f"Discussion score {r.exercise_score:.1f}; completeness {r.completeness_pct:.1f}%"}
    for r in ranked.head(3).itertuples()])
display(log)
from fieldguide import download_link
display(download_link(log.to_csv(index=False),"exercise-decision-log.csv","text/csv"))

## Practice and debrief

Write an alternative prioritization rule without a composite score. Compare its top three districts with the weighted result. State an equity concern and what information could change your recommendation. Record the owner, review time and evidence needed in the decision log.

## Sources and further learning

[WHO public health EOC framework](https://www.who.int/publications/i/item/framework-for-a-public-health-emergency-operations-centre) · [WHO emergency operations](https://www.who.int/emergencies/operations).